# 04 — Luồng S1: khám phá lớp mới (atlas)

**Mục đích.** Theo dữ liệu qua phiên S1, với đầu vào và đầu ra thật của từng module: M10 → M11 → M7 (F) + M8 → M9 + M12 → M7 (S) + M8 → M9 + M13 (mã M ở `notes/nb/README.vi.md`). Chốt giao diện của M10 (nhận gì, trả gì) cho khoang `12_cluster` và script tầng 1.

**Phạm vi.**
- GC10, đặc trưng của cấu hình paper: fine-tune trên tập có nhãn, FT seed 0 (`gc10_ft_cls5first_img80pct_r0`); giao thức t1 + `--hungarian_on eval` + S0 có nhãn. Bộ không fine-tune (`fz`) là bối cảnh phụ.
- Seed 4: mode đa số của bộ này (New 57.0, cùng nhóm seed 0, 1, 3; seed 2 lệch, 49.1).
- Atlas mỏng: chỉ đầu vào → hàm → đầu ra. Mổ sâu và cấy ghép nằm ở khoang 12 (M10) và 13 (M12).
- Số trong notebook chỉ để minh hoạ; số chính thức lấy từ `exp.py`.

In [ ]:
# cell 2: provenance (run first)
import os, subprocess, sys
from pathlib import Path
ROOT = Path.cwd().resolve()
while not (ROOT / "main.py").exists() and ROOT != ROOT.parent:
    ROOT = ROOT.parent
os.chdir(ROOT); sys.path[:0] = [str(ROOT), str(ROOT / "scripts")]
git = lambda *a: subprocess.run(["git", *a], capture_output=True, text=True).stdout.strip()
print("commit", git("rev-parse", "--short", "HEAD"),
      "| code dirty", bool(git("status", "--porcelain", "--", ".", ":(exclude)runs", ":(exclude)notes")),
      "| features", Path("runs/inputs.txt").read_text().strip())

commit 305481a | code dirty False | features features/20261005_013042__gc10_fz_cls5first_img80pct/dino-vitb16


In [3]:
# cell 3: replay seed 4 on the paper-setting features (fine-tuned, FT seed 0), check against the pipeline run
import contextlib, io
from pathlib import Path
from replay import replay, _reference
from extract_vbcgcd import clean_lines, extract

SEED, FT = 4, "gc10_ft_cls5first_img80pct_r0"
fdir = sorted(Path("features").glob(f"*__{FT}"))
assert len(fdir) == 1, fdir
FEATURES = fdir[0] / "dino-vitb16-sl"
buf = io.StringIO()
with contextlib.redirect_stdout(buf), contextlib.redirect_stderr(buf):
    r = replay(SEED, features=FEATURES)
log = buf.getvalue()                                      # full console output, kept for later cells
m = extract(clean_lines(log))                             # same parser as the official runs

_, _, ref = _reference(SEED, tag=f"{FT}_km_evhung_d64")   # group-A phase-2 run, same seed
st = r["stages"][1]
acc = {k: (round(st["acc"][k], 2), round(ref[f"S1_{k}"], 2)) for k in ("All", "Old", "New", "S0test")}
same_acc = all(st["acc"][k] == ref[f"S1_{k}"] for k in acc)
print("features:", FEATURES)
print("S1 accuracy (replay, run):", acc, "->", "SAME" if same_acc else "DIFF")
print("parsed from the log:", {k: v for k, v in m.items() if k.startswith(("S0_All", "S1_novel", "S1_estop"))})
assert same_acc, "replay differs from the pipeline run: stop here"

features: features/20261005_013115__gc10_ft_cls5first_img80pct_r0/dino-vitb16-sl
S1 accuracy (replay, run): {'All': (70.11, 70.11), 'Old': (81.74, 81.74), 'New': (57.01, 57.01), 'S0test': (81.74, 81.74)} -> SAME
parsed from the log: {'S0_All': 85.47717842323651, 'S1_novel_kept': 689, 'S1_estop_S': 526}


In [5]:
# cell 4 - M10 clustering: in = S1 pool (N, 768) + K; out = cluster id per image (N,)
# paper: Alg. 1 lines 25-27, App. C.3 (k-means); code: clustering/kmeans.py KMeansCluster; D5, D18
import json
import numpy as np
import pandas as pd

CLASSES = json.load(open("splits/gc10_split.json"))["classes"]
NAMES = [c.split("_")[0] for c in CLASSES]                         # short names; ids 0-4 base, 5-9 novel
pool, ids = st["train"], st["raw"]
y = np.asarray(pool._y).astype(int)                                # true labels: scoring only
print("in :", pool._x.shape, "| K =", r["args"].increment, "|", type(st["cluster"]).__name__,
      "n_init =", st["cluster"].model.n_init, "| inertia =", st["inertia"])
print("out:", ids.shape, "values", sorted(set(ids.tolist())), "| cluster sizes", np.bincount(ids).tolist())
xt = pd.crosstab(pd.Series([NAMES[c] for c in y], name="true class"), pd.Series(ids, name="cluster"))
xt = xt.reindex([n for n in NAMES if n in xt.index])              # rows in class-id order
expected = {"silk": 420, "inclusion": 135, "rolled": 17, "crease": 32, "waist": 94,
            **{n: 10 for n in NAMES[:5]}}                          # pool composition (PL-E01 check I2)
assert xt.sum(axis=1).to_dict() == expected, xt.sum(axis=1).to_dict()
xt

in : (748, 768) | K = 5 | KMeansCluster n_init = 10 | inertia = 82926.203125
out: (748,) values [0, 1, 2, 3, 4] | cluster sizes [95, 157, 272, 121, 103]


cluster,0,1,2,3,4
true class,,,,,
punching,0,10,0,0,0
welding,1,9,0,0,0
crescent,6,4,0,0,0
water,3,0,6,1,0
oil,6,0,4,0,0
silk,29,1,169,118,103
inclusion,46,11,76,2,0
rolled,1,16,0,0,0
crease,3,27,2,0,0


In [6]:
# cell 5 - M11 cluster naming, EVALUATION ONLY: in = cluster ids of the novel test images + true test labels;
# out = pi (cluster -> novel class) and lut (true id -> cluster-space id, used to relabel test labels)
# paper: no counterpart (D13: upstream named clusters with TRUE train labels; here only at evaluation)
te, off = st["test"], st["offset"]
yt = np.asarray(te._y).astype(int)                                 # novel test labels (absolute ids)
ct = st["cluster"].predict_raw(te._x)                              # cluster id of each novel test image
xt_test = pd.crosstab(pd.Series([NAMES[c] for c in yt], name="true class (test)"), pd.Series(ct, name="cluster"))
xt_test = xt_test.reindex([n for n in NAMES if n in xt_test.index])
print("in : test", te._x.shape, "| pi_test (cluster -> novel class):", st["pi_test"])
owner_pool = xt.idxmax(axis=0)                                     # plurality class of each cluster in the pool
for c in range(r["args"].increment):
    name = NAMES[off + st["pi_test"][c]]
    hit = xt_test.loc[name, c] if c in xt_test.columns else 0
    print(f"cluster {c}: named {name:9s} | pool owner {owner_pool[c]:9s} | test hits {hit}/{(yt == off + st['pi_test'][c]).sum()}")
print("lut (true id -> cluster-space id):", st["lut"].tolist())
xt_test

in : test (214, 768) | pi_test (cluster -> novel class): {2: 0, 0: 1, 3: 2, 4: 3, 1: 4}
cluster 0: named inclusion | pool owner inclusion | test hits 15/43
cluster 1: named waist     | pool owner waist     | test hits 28/30
cluster 2: named silk      | pool owner silk      | test hits 86/124
cluster 3: named rolled    | pool owner silk      | test hits 0/6
cluster 4: named crease    | pool owner silk      | test hits 0/11
lut (true id -> cluster-space id): [0, 1, 2, 3, 4, 7, 5, 8, 9, 6]


cluster,0,1,2,3,4
true class (test),,,,,
silk,4,4,86,21,9
inclusion,15,8,20,0,0
rolled,0,6,0,0,0
crease,4,7,0,0,0
waist,1,28,1,0,0


In [10]:
# cell 6 - M7 SVI, F pass (+ M8 early stop): in = pool in PCA space (N, d) + cluster-space labels (raw + offset);
# out = mu', Sigma' for all C slots. Slots 0-4 = base classes; slots 5-9 = clusters 0-4 (cluster order, D13 eval mode)
# paper: Alg. 1 lines 29-33, Eq. 7, Eq. 11; D11, D14, D23
import jax, jax.numpy as jnp

clf, K, base = r["clf"], r["args"].increment, r["args"].base
SLOT = NAMES[:off] + [f"cl{c}={NAMES[off + st['pi_test'][c]]}" for c in range(K)]   # cluster c, eval-time name
Xp, _ = clf.pre_processing(pool._x, st["raw"] + off)                 # M6 reused: the PCA fitted at S0
(n0, p0), (nF, pF) = r["svi"][0], r["svi"][1]                         # parameters after S0 and after the F pass
mu0, muF = np.asarray(p0["class_means"]), np.asarray(pF["class_means"])
S0c, SFc = np.asarray(p0["class_covs"]), np.asarray(pF["class_covs"])
print("in :", Xp.shape, "| labels", sorted(set((st["raw"] + off).tolist())), "|", n0, "->", nF)
print("out:", muF.shape, SFc.shape)
print(pd.DataFrame({"slot": SLOT,
                    "|mu_F - mu_S0|": np.linalg.norm(muF - mu0, axis=1).round(4),
                    "logdet Sigma_S0": np.linalg.slogdet(S0c.astype(np.float64))[1].round(3),
                    "logdet Sigma_F": np.linalg.slogdet(SFc.astype(np.float64))[1].round(3),
                    "min eig Sigma_F": np.linalg.eigvalsh(SFc.astype(np.float64)).min(axis=1).round(4)}).to_string(index=False))
det = lambda S: float(jax.vmap(jnp.linalg.det)(jnp.asarray(S)).mean())      # float32, as in the code
d_old, d_new, sf = det(S0c[:base]), det(SFc[off:off + K]), (off + K) / base  # F pass: use_correct_scaling_factor=False
stop = (d_old > 1 and d_new > sf * d_old) or (d_old < 1 and d_new < d_old / sf)
print(f"M8 at the end of F: det_old={d_old:.4g}  det_new={d_new:.4g}  scaling={sf:g}  -> stop condition {stop}")
print("F-pass early stop in the log:", m.get("S1_estop_F", f"none (full budget, {r['args'].n_epochs} steps)"))

in : (748, 64) | labels [5, 6, 7, 8, 9] | stage_0_Flearning -> stage_1_Flearning
out: (10, 64) (10, 64, 64)
         slot  |mu_F - mu_S0|  logdet Sigma_S0  logdet Sigma_F  min eig Sigma_F
     punching          0.0000           -1.417          -1.417           0.8241
      welding          0.0000           -0.838          -0.838           0.8315
     crescent          0.0000           -1.135          -1.135           0.8315
        water          0.0000           -0.866          -0.866           0.8268
          oil          0.0000           -0.829          -0.829           0.8436
cl0=inclusion          0.1366            0.000          -1.189           0.8524
    cl1=waist          0.1650            0.000          -0.971           0.8493
     cl2=silk          0.1440            0.000          -1.882           0.7771
   cl3=rolled          0.1579            0.000          -1.737           0.8081
   cl4=crease          0.1609            0.000          -1.965           0.8189
M8 at the en

In [12]:
# cell 6b (optional) - how far the learned means are from the data: r_mu = ||mu_k - xbar_k|| / ||xbar_k||  (D23)
lab = st["raw"] + off
xbar = np.stack([Xp[lab == k].mean(0) for k in range(off, off + K)])
r_mu = np.linalg.norm(muF[off:off + K] - xbar, axis=1) / np.linalg.norm(xbar, axis=1)
print("||xbar_k||:", np.linalg.norm(xbar, axis=1).round(3), "| ||mu_F||:", np.linalg.norm(muF[off:off + K], axis=1).round(3),
      "| r_mu:", r_mu.round(3))

||xbar_k||: [ 9.212  6.558 10.38  13.023 13.141] | ||mu_F||: [0.137 0.165 0.144 0.158 0.161] | r_mu: [0.991 0.98  0.99  0.991 0.991]


In [13]:
# cell 7 - M9 classify the pool with the F-pass parameters + M12 old/new filter:
# in = pool (N, d) + mu', Sigma'; out = keep mask (N,); kept images keep their CLUSTER labels (D12: the paper re-labels)
# paper: Alg. 1 lines 34-36; code: mngmm.train (novel_idx = pred >= offset)
pred, _ = clf._predict(jnp.asarray(Xp), pF)
pred = np.asarray(pred)
keep = pred >= off                                                   # predicted as a novel class -> kept
assert keep.sum() == m["S1_novel_kept"], (keep.sum(), m["S1_novel_kept"])   # same count as the pipeline log
is_old = y < off                                                     # true labels: scoring only
print(f"kept {keep.sum()} / {len(keep)} | old images kept {(keep & is_old).sum()} / {is_old.sum()}"
      f" | novel images dropped {(~keep & ~is_old).sum()} / {(~is_old).sum()}")
print(pd.DataFrame({"class": NAMES, "in pool": np.bincount(y, minlength=10),
                    "dropped": np.bincount(y[~keep], minlength=10)}).to_string(index=False))

kept 689 / 748 | old images kept 2 / 50 | novel images dropped 11 / 698
    class  in pool  dropped
 punching       10       10
  welding       10       10
 crescent       10       10
    water       10        8
      oil       10       10
     silk      420        0
inclusion      135        8
   rolled       17        2
   crease       32        0
    waist       94        1


In [14]:
# cell 8 - M7 SVI, S pass (+ M8): in = kept pool images (n, d) with their CLUSTER labels; out = final mu, Sigma
# the S pass restarts from the S0 parameters (svi.init uses global_params, updated only after S): F only feeds the filter
# paper: Alg. 1 lines 37-40, Eq. 11; D12, D14, D23
nS, pS = r["svi"][2]
muS, SSc = np.asarray(pS["class_means"]), np.asarray(pS["class_covs"])
print("in :", int(keep.sum()), "kept images |", nF, "->", nS, "| out:", muS.shape, SSc.shape)
print(pd.DataFrame({"slot": SLOT,
                    "|mu_S|": np.linalg.norm(muS, axis=1).round(4),
                    "|mu_F|": np.linalg.norm(muF, axis=1).round(4),
                    "|mu_S - mu_F|": np.linalg.norm(muS - muF, axis=1).round(4),
                    "logdet Sigma_S": np.linalg.slogdet(SSc.astype(np.float64))[1].round(3)}).to_string(index=False))
sfS = float(np.sqrt(K / (off + K)))                                  # S pass: use_correct_scaling_factor=True
d_old, d_new = det(S0c[:off]), det(SSc[off:off + K])
print(f"M8 for S: stop when det_new < det_old / {sfS:.3f} = {d_old / sfS:.4g} | at the end: det_new={d_new:.4g}"
      f" | stopped at step {m.get('S1_estop_S', 'none')}")

in : 689 kept images | stage_1_Flearning -> stage_1_Slearning | out: (10, 64) (10, 64, 64)
         slot  |mu_S|  |mu_F|  |mu_S - mu_F|  logdet Sigma_S
     punching  0.0860  0.0860         0.0000          -1.417
      welding  0.0913  0.0913         0.0000          -0.838
     crescent  0.0735  0.0735         0.0000          -1.135
        water  0.0904  0.0904         0.0000          -0.866
          oil  0.1006  0.1006         0.0000          -0.829
cl0=inclusion  0.0885  0.1366         0.0611          -0.537
    cl1=waist  0.0998  0.1650         0.0725          -0.345
     cl2=silk  0.0915  0.1440         0.0553          -0.816
   cl3=rolled  0.0952  0.1579         0.0639          -0.764
   cl4=crease  0.0958  0.1609         0.0661          -0.860
M8 for S: stop when det_new < det_old / 0.707 = 0.5242 | at the end: det_new=0.5248 | stopped at step 526


In [15]:
# cell 9 - M13 evaluation: in = final parameters + test sets (labels relabeled into cluster space by lut);
# out = All / Old / New / S0test, plus per-class recall on the novel test set. paper: App. C.2; D13, D15
print("S1 accuracy:", {k: round(v, 2) for k, v in st["acc"].items()})
Xt, _ = clf.pre_processing(te._x, yt)
pt = np.asarray(clf._predict(jnp.asarray(Xt), pS)[0])                # final prediction over all 10 slots
lut = st["lut"]
assert np.isclose(100 * (pt == lut[yt]).mean(), st["acc"]["New"]), "recomputed New differs from the pipeline"
rows = []
for c in range(off, off + K):
    sel = yt == c
    named = [k for k, t in st["pi_test"].items() if t == c - off][0]   # cluster carrying class c's name
    rows.append({"class": NAMES[c], "test n": int(sel.sum()),
                 "recall after M10/M11": round(100 * (ct[sel] == named).mean(), 1),
                 "recall final": round(100 * (pt[sel] == lut[c]).mean(), 1),
                 "-> old slots": int((pt[sel] < off).sum()),
                 "-> other novel slots": int(((pt[sel] >= off) & (pt[sel] != lut[c])).sum())})
tab = pd.DataFrame(rows)
print(tab.to_string(index=False))
print("balanced novel: after M10/M11", round(tab["recall after M10/M11"].mean(), 1), "| final", round(tab["recall final"].mean(), 1))

S1 accuracy: {'All': 70.11, 'Old': 81.74, 'New': 57.01, 'S0test': 81.74}
    class  test n  recall after M10/M11  recall final  -> old slots  -> other novel slots
     silk     124                  69.4          71.0             0                    36
inclusion      43                  34.9          14.0            22                    15
   rolled       6                   0.0           0.0             3                     3
   crease      11                   0.0           9.1             6                     4
    waist      30                  93.3          90.0             0                     3
balanced novel: after M10/M11 39.5 | final 36.8


## Paper ↔ code (luồng S1)

- M10 phân cụm: paper dùng k-means (C.3), code upstream dùng GMM (D5); ở đây k-means, `n_init` 10 (D18).
- M11 đặt tên cụm: paper không có bước này; upstream đặt tên bằng nhãn thật của train (D13); ở đây chỉ dùng lúc đánh giá.
- M7 lượt F và lượt S: paper là SVI với ELBO (Eq. 7); code tối ưu likelihood, không có prior (D11). Mỗi lần gọi đều khởi tạo lại từ tham số của các phiên trước, nên lượt S không nối tiếp lượt F (khớp Alg. 1 dòng 2). Tham số gần như không rời điểm khởi tạo (D23; cell 6b).
- M8 dừng sớm: paper dùng Eq. 11; code so det trung bình của lớp mới với lớp cũ (D14). Điểm dừng do tốc độ co của Σ quyết định (cell 8).
- M12: paper gán lại nhãn (Alg. 1 dòng 34–36); code chỉ lọc và giữ nhãn cụm (D12).
- M13: M_f là đại lượng thay thế (D15).

## Nghẽn (một seed; số chính thức lấy từ run)

- M10 (khoang 12): rolled và crease không có cụm riêng; recall 0 ngay sau phân cụm, mất 40 điểm balanced novel (PL-Q1, PL-E01).
- M9 với 10 slot (khoang 11): 31/214 ảnh test lớp mới bị đoán thành lớp cũ, 22 trong số đó là inclusion (PL-Q8).
- M12 (khoang 13): bộ lọc loại 48/50 ảnh cũ; mất 11/698 ảnh mới, trong đó 2/17 rolled (PL-Q3).

## Summary (EN)

Single-seed walk-through on the paper setting (fine-tuned features, FT seed 0, seed 4; run `gc10_ft_cls5first_img80pct_r0_km_evhung_d64_s4`, reproduced exactly by `scripts/replay.py`). Illustrative; official numbers come from the runs.

1. k-means splits silk_spot over three clusters and puts 27/32 crease and 16/17 rolled_pit pool images into the waist_folding cluster; at evaluation the two rare classes receive the names of two silk clusters and get 0% test recall after clustering.
2. Clustering alone leaves 39.5% balanced novel accuracy on the test set (at most 60% once two of five novel classes own no cluster); the full pipeline ends at 36.8% (New 57.01%).
3. The SVI passes barely move the Gaussians: learned novel means cover 1–2% of the distance to the data means (r_mu 0.98–0.99); the S pass restarts from the S0 parameters, so the F pass only feeds the old/new filter, and the determinant rule stops the S pass at step 526 of 1000.
4. The old/new filter removes 48/50 old pool images and drops 11/698 novel ones (2/17 rolled_pit); the final 10-way classifier sends 31/214 novel test images to old-class slots, 22 of them inclusion.